# Part 1. Set up the notebook

### Step 1: Enable the GPU runtime

1. In the Colab menu, click Runtime → Change runtime type
2. Under "Hardware accelerator", select T4 GPU
3. Click Save

### Step 2: Install packages

1. Run the following cell
2. A pop up window should appear, select "Restart session"
3. Rerun the cell and move on to the next one



In [ ]:
!pip install git+https://github.com/mariannafoschi/kine.git
!pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install ehtim finufft flax optax

  Cloning https://github.com/mariannafoschi/kine.git to /tmp/pip-req-build-4r26q8w1
  Running command git clone --filter=blob:none --quiet https://github.com/mariannafoschi/kine.git /tmp/pip-req-build-4r26q8w1
  Resolved https://github.com/mariannafoschi/kine.git to commit 9aee518fd6b7e6cf47221596bcfa6338161fbbe6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


In [ ]:
import jax
print("JAX devices:", jax.devices())
print(jax.numpy.ones(3))

JAX devices: [CudaDevice(id=0)]
[1. 1. 1.]


### Step 3: Import packages

Ensure all packages load correctly

In [ ]:
import glob
import queue
import warnings
import numpy as np
import ehtim as eh
import ehtim.const_def as ehc
from tqdm.notebook import tqdm
from collections import OrderedDict as odict

import jax
import optax
from flax import linen as nn
from jax import numpy as jnp

import kine.model as mo
import kine.obsdata as ob
import kine.trainer as tr
import kine.utils as ut
import kine.video as vi

print("✓ All imports successful")
print("✓ GPU ready for kine")

Welcome to eht-imaging!


✓ All imports successful
✓ GPU ready for kine


**Note:** GPU-accelerated NUFFT is not available in colab but it is implemented in the code. It is designed for multi-epoch imaging of large datasets.

# Part 2. Upload observations and set imaging parameters
### Step 1: Upload observation file to Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving hops_2017_M87.uvfits to hops_2017_M87.uvfits


### Step 2: Set imaging parameters
**Note:** In the actual code this is done by loading a .yaml file, but for this Colab tutorial we will set them directly in the notebook.

In [ ]:
# Data pre-processing
# -----------------------------
# Time averaging
tavg = 60
# Systematic noise
syserr = 0.01
# Minimum number of baselines per data snapshot
min_bl = 0

# Coordinates and data products
# -----------------------------
# Field of view (uas)
fov_uas = 200
# Space resolution (pix)
npix = 64
# Data products
data_prod = ['cphaseI', 'logcampI']

# Network initialization
# -----------------------------
init_params = {
  'fwhm': 80,
  'blur': 20,
  'posx': 0,
  'posy': 0
}
# Training
# -----------------------------
# Initialization seed
seed = 1
# Number of epochs
niter = 3000
initniter = 2000
# Degrees of positional encoding ([t, x, y])
nposenc = [0, 0]
# Network parameters
depth = 4
width = 256
outshift = 10
scaling_i = 1

# Part 3. Run the algorithm
### Step 1: Load and preprocess the observations